In [2]:
import findspark
findspark.init()
from pyspark.sql import functions as f, SparkSession, types as t
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql.functions import udf, col, monotonically_increasing_id
from datetime import datetime, timedelta
import os

path_jar_driver = os.environ.get('CONNECTOR_J_PATH', 
    '/workspaces/Modelado-de-Datos-y-ETL/.devcontainer/mysql-connector-j-9.3.0.jar')

conf = SparkConf().set('spark.driver.extraClassPath', path_jar_driver)
spark_context = SparkContext(conf=conf)
sql_context = SQLContext(spark_context)
spark = sql_context.sparkSession

source_db = 'jdbc:mysql://157.253.236.120:8080/WWImportersTransactional'
dest_db   = 'jdbc:mysql://157.253.236.120:8080/DB_202613_n_cantini'
db_user   = 'DB_202613_n_cantini'
db_psswd  = '201630737'

def obtener_df(sql):
    return spark.read.format('jdbc') \
        .option('url', source_db) \
        .option('dbtable', sql) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .load()

def guardar_df(df, tabla):
    df.write.format('jdbc') \
        .mode('append') \
        .option('url', dest_db) \
        .option('dbtable', tabla) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .save()

print("Ambiente configurado correctamente")

Picked up JAVA_TOOL_OPTIONS: -Xss512k -XX:+UseContainerSupport
Picked up JAVA_TOOL_OPTIONS: -Xss512k -XX:+UseContainerSupport
26/06/30 18:55:42 WARN Utils: Your hostname, codespaces-7ba9a3 resolves to a loopback address: 127.0.0.1; using 10.0.10.47 instead (on interface eth0)
26/06/30 18:55:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/30 18:55:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Ambiente configurado correctamente


/home/vscode/.local/lib/python3.11/site-packages/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [3]:
# BLOQUE 1 - DIMENSIÓN PROVEEDOR
# EXTRACCIÓN
sql_proveedores = '''(SELECT p.ProveedorID, p.NombreProveedor, 
       c.CategoriaProveedor,
       per.NombreCompleto AS ContactoPrincipal,
       p.ReferenciaProveedor,
       p.DiasPago,
       p.CodigoPostal
FROM proveedoresCopia p
JOIN CategoriasProveedores c ON p.CategoriaProveedorID = c.CategoriaProveedorID
JOIN Personas per ON p.PersonaContactoPrincipalID = per.ID_persona
WHERE p.ProveedorID IN (1,2,3,4,5,6,7,8,9,10,11,12,13)) AS Temp_proveedores'''

proveedores = obtener_df(sql_proveedores)
proveedores.show(15)

+-----------+--------------------+--------------------+------------------+-------------------+--------+------------+
|ProveedorID|     NombreProveedor|  CategoriaProveedor| ContactoPrincipal|ReferenciaProveedor|DiasPago|CodigoPostal|
+-----------+--------------------+--------------------+------------------+-------------------+--------+------------+
|          1| A Datum Corporation| productos novedosos|        Reio Kabin|            AA20384|     -14|       46077|
|          2|        Contoso Ltd.| productos novedosos|   Hanna Mihhailov|           B2084020|      -7|       98253|
|          3|Consolidated Mess...|servicios de mens...|      Kerstin Parn|          209340283|     -30|       94101|
|          4|       Fabrikam Inc.|                ropa|       Bill Lawson|             293092|      30|       40351|
|          5|Graphic Design In...| productos novedosos|        Penny Buck|            8803922|      14|       64847|
|          6| Humongous Insurance|servicios de seguros|Madelaine

In [4]:
# TRANSFORMACIÓN
# 1. Corregir DiasPago negativos multiplicando por -1
from pyspark.sql.functions import abs as spark_abs

proveedores = proveedores.withColumn('DiasPago', spark_abs(col('DiasPago')))

# 2. Renombrar columnas con convención DWH
proveedores = proveedores.selectExpr(
    'ProveedorID AS ID_Proveedor_T',
    'NombreProveedor AS Nombre',
    'CategoriaProveedor AS Categoria',
    'ContactoPrincipal',
    'ReferenciaProveedor AS Referencia',
    'DiasPago',
    'CodigoPostal'
)

# 3. Crear ID propio de la bodega
proveedores = proveedores.coalesce(1).withColumn(
    'ID_Proveedor_DWH', monotonically_increasing_id() + 1
)

# 4. Ordenar columnas
proveedores = proveedores.select(
    'ID_Proveedor_DWH', 'ID_Proveedor_T', 'Nombre',
    'Categoria', 'ContactoPrincipal', 'Referencia',
    'DiasPago', 'CodigoPostal'
)

proveedores.show(15)

+----------------+--------------+--------------------+--------------------+------------------+-----------+--------+------------+
|ID_Proveedor_DWH|ID_Proveedor_T|              Nombre|           Categoria| ContactoPrincipal| Referencia|DiasPago|CodigoPostal|
+----------------+--------------+--------------------+--------------------+------------------+-----------+--------+------------+
|               1|             1| A Datum Corporation| productos novedosos|        Reio Kabin|    AA20384|      14|       46077|
|               2|             2|        Contoso Ltd.| productos novedosos|   Hanna Mihhailov|   B2084020|       7|       98253|
|               3|             3|Consolidated Mess...|servicios de mens...|      Kerstin Parn|  209340283|      30|       94101|
|               4|             4|       Fabrikam Inc.|                ropa|       Bill Lawson|     293092|      30|       40351|
|               5|             5|Graphic Design In...| productos novedosos|        Penny Buck|   

In [5]:
# CARGUE
guardar_df(proveedores, 'DB_202613_n_cantini.M_Proveedor')
print("M_Proveedor cargada correctamente")

M_Proveedor cargada correctamente


In [7]:
# BLOQUE 2 - DIMENSIÓN TIPO TRANSACCIÓN
# EXTRACCIÓN
sql_tipos = '''(SELECT TipoTransaccionID, TipoTransaccionNombre
               FROM TiposTransaccion) AS Temp_tipos'''

tipos = obtener_df(sql_tipos)
tipos.show()

+-----------------+---------------------+
|TipoTransaccionID|TipoTransaccionNombre|
+-----------------+---------------------+
|                2| Customer Credit Note|
|                3| Customer Payment ...|
|                4|      Customer Refund|
|                5|     Supplier Invoice|
|                6| Supplier Credit Note|
|                7| Supplier Payment ...|
|                8|      Supplier Refund|
|                9|       Stock Transfer|
|               10|          Stock Issue|
|               11|        Stock Receipt|
|               12| Stock Adjustment ...|
|               13|      Customer Contra|
+-----------------+---------------------+



In [8]:
# TRANSFORMACIÓN
tipos = tipos.selectExpr(
    'TipoTransaccionID AS ID_TipoTransaccion_T',
    'TipoTransaccionNombre AS Tipo'
)

tipos = tipos.coalesce(1).withColumn(
    'ID_TipoTransaccion_DWH', monotonically_increasing_id() + 1
)

tipos = tipos.select(
    'ID_TipoTransaccion_DWH', 'ID_TipoTransaccion_T', 'Tipo'
)

tipos.show()

# CARGUE
guardar_df(tipos, 'DB_202613_n_cantini.M_TipoTransaccion')
print("M_TipoTransaccion cargada correctamente")

+----------------------+--------------------+--------------------+
|ID_TipoTransaccion_DWH|ID_TipoTransaccion_T|                Tipo|
+----------------------+--------------------+--------------------+
|                     1|                   2|Customer Credit Note|
|                     2|                   3|Customer Payment ...|
|                     3|                   4|     Customer Refund|
|                     4|                   5|    Supplier Invoice|
|                     5|                   6|Supplier Credit Note|
|                     6|                   7|Supplier Payment ...|
|                     7|                   8|     Supplier Refund|
|                     8|                   9|      Stock Transfer|
|                     9|                  10|         Stock Issue|
|                    10|                  11|       Stock Receipt|
|                    11|                  12|Stock Adjustment ...|
|                    12|                  13|     Customer Con

M_TipoTransaccion cargada correctamente


In [9]:
# BLOQUE 3 - DIMENSIÓN FECHA
# EXTRACCIÓN - obtener rango de fechas de movimientos
sql_fechas = '''(SELECT MIN(FechaTransaccion) AS fecha_min, 
                        MAX(FechaTransaccion) AS fecha_max
                FROM movimientosCopia) AS Temp_fechas'''

rango = obtener_df(sql_fechas)
rango.show()

+--------------------+-----------+
|           fecha_min|  fecha_max|
+--------------------+-----------+
|2013-12-31 07:00:...|Sep 30,2015|
+--------------------+-----------+



In [10]:
# TRANSFORMACIÓN - generar dimensión fecha
from datetime import datetime, timedelta

# Definir rango manualmente basado en lo que vimos
fecha_inicio = datetime(2013, 12, 31)
fecha_fin = datetime(2015, 9, 30)

# Generar lista de fechas
fechas = []
fecha_actual = fecha_inicio
id_fecha = 1

while fecha_actual <= fecha_fin:
    fechas.append((
        id_fecha,
        fecha_actual.strftime('%Y-%m-%d'),
        fecha_actual.day,
        fecha_actual.strftime('%B'),  # Nombre del mes en inglés
        fecha_actual.year,
        int(fecha_actual.strftime('%V'))  # Número semana ISO
    ))
    fecha_actual += timedelta(days=1)
    id_fecha += 1

# Crear dataframe
schema_fecha = ['ID_Fecha', 'Fecha', 'Dia', 'Mes', 'Anio', 'Numero_semana_ISO']
df_fecha = spark.createDataFrame(fechas, schema=schema_fecha)
df_fecha.show(5)
print("Total fechas generadas:", df_fecha.count())

# CARGUE
guardar_df(df_fecha, 'DB_202613_n_cantini.M_Fecha')
print("M_Fecha cargada correctamente")

+--------+----------+---+--------+----+-----------------+
|ID_Fecha|     Fecha|Dia|     Mes|Anio|Numero_semana_ISO|
+--------+----------+---+--------+----+-----------------+
|       1|2013-12-31| 31|December|2013|                1|
|       2|2014-01-01|  1| January|2014|                1|
|       3|2014-01-02|  2| January|2014|                1|
|       4|2014-01-03|  3| January|2014|                1|
|       5|2014-01-04|  4| January|2014|                1|
+--------+----------+---+--------+----+-----------------+
only showing top 5 rows

Total fechas generadas: 639


M_Fecha cargada correctamente


In [12]:
# BLOQUE 4 - TABLA DE HECHOS MOVIMIENTOS
# EXTRACCIÓN
sql_movimientos = '''(SELECT * FROM movimientosCopia LIMIT 5) AS Temp_movimientos'''

movimientos = obtener_df(sql_movimientos)
movimientos.printSchema()

root
 |-- TransaccionProductoID: integer (nullable = true)
 |-- ProductoID: integer (nullable = true)
 |-- TipoTransaccionID: integer (nullable = true)
 |-- ClienteID: double (nullable = true)
 |-- InvoiceID: double (nullable = true)
 |-- ProveedorID: string (nullable = true)
 |-- OrdenDeCompraID: string (nullable = true)
 |-- FechaTransaccion: string (nullable = true)
 |-- Cantidad: double (nullable = true)



In [13]:
# EXTRACCIÓN VF
sql_movimientos = '''(SELECT TransaccionProductoID, ProductoID, 
                             TipoTransaccionID, ClienteID, 
                             InvoiceID, ProveedorID, 
                             OrdenDeCompraID, FechaTransaccion, 
                             Cantidad
                      FROM movimientosCopia) AS Temp_movimientos'''

movimientos = obtener_df(sql_movimientos)
print("Total registros:", movimientos.count())
print("Registros distintos:", movimientos.distinct().count())
movimientos.show(5)

Total registros: 204292


Registros distintos: 173659


+---------------------+----------+-----------------+---------+---------+-----------+---------------+----------------+--------+
|TransaccionProductoID|ProductoID|TipoTransaccionID|ClienteID|InvoiceID|ProveedorID|OrdenDeCompraID|FechaTransaccion|Cantidad|
+---------------------+----------+-----------------+---------+---------+-----------+---------------+----------------+--------+
|               118903|       217|               10|    476.0|  24904.0|           |               |     Apr 25,2014|   -40.0|
|               286890|       135|               10|     33.0|  60117.0|           |               |     Dec 10,2015|    -7.0|
|               285233|       111|               10|    180.0|  59768.0|           |               |     Dec 04,2015|    -2.0|
|               290145|       213|               10|     33.0|  60795.0|           |               |     Dec 23,2015|    -3.0|
|               247492|        90|               10|     55.0|  51851.0|           |               |     Jul 27

In [14]:
# TRANSFORMACIÓN
from pyspark.sql.functions import to_date, when, regexp_replace

# 1. Eliminar duplicados totales
movimientos = movimientos.distinct()
print("Registros tras eliminar duplicados:", movimientos.count())

# 2. Estandarizar fechas a YYYY-MM-DD
regex = "([0-2]\\d{3}-(0[1-9]|1[0-2])-(0[1-9]|[1-2][0-9]|3[0-1]))"
cumplen = movimientos.filter(movimientos["FechaTransaccion"].rlike(regex))
no_cumplen = movimientos.filter(~movimientos["FechaTransaccion"].rlike(regex))

print("Fechas con formato correcto:", cumplen.count())
print("Fechas con formato incorrecto:", no_cumplen.count())

no_cumplen = no_cumplen.withColumn(
    'FechaTransaccion',
    f.udf(lambda d: datetime.strptime(d, '%b %d,%Y').strftime('%Y-%m-%d'), 
          t.StringType())(f.col('FechaTransaccion'))
)

movimientos = cumplen.union(no_cumplen)
print("Total tras estandarizar fechas:", movimientos.count())
movimientos.show(5)

Registros tras eliminar duplicados: 173659


Fechas con formato correcto: 109405


Fechas con formato incorrecto: 64254


Total tras estandarizar fechas: 173659


+---------------------+----------+-----------------+---------+---------+-----------+---------------+--------------------+--------+
|TransaccionProductoID|ProductoID|TipoTransaccionID|ClienteID|InvoiceID|ProveedorID|OrdenDeCompraID|    FechaTransaccion|Cantidad|
+---------------------+----------+-----------------+---------+---------+-----------+---------------+--------------------+--------+
|               284303|        69|               10|    976.0|  59568.0|           |               |2015-12-02 12:00:...|   -10.0|
|               175408|       210|               10|    894.0|  36730.0|           |               |2014-11-19 12:00:...|   -10.0|
|               175498|        26|               10|    173.0|  36750.0|           |               |2014-11-19 12:00:...|   -10.0|
|               273388|        65|               10|    111.0|  57285.0|           |               |2015-10-23 12:00:...|   -10.0|
|               235320|         9|               10|    864.0|  49272.0|           

In [15]:
# 3. Limpiar hora de las fechas que la tienen
movimientos = movimientos.withColumn(
    'FechaTransaccion',
    f.udf(lambda d: d[:10] if d else d, t.StringType())(f.col('FechaTransaccion'))
)

# 4. Verificar fechas mínima y máxima
movimientos.agg(
    f.min('FechaTransaccion').alias('fecha_min'),
    f.max('FechaTransaccion').alias('fecha_max')
).show()

movimientos.show(5)

+----------+----------+
| fecha_min| fecha_max|
+----------+----------+
|2013-12-31|2016-05-31|
+----------+----------+



+---------------------+----------+-----------------+---------+---------+-----------+---------------+----------------+--------+
|TransaccionProductoID|ProductoID|TipoTransaccionID|ClienteID|InvoiceID|ProveedorID|OrdenDeCompraID|FechaTransaccion|Cantidad|
+---------------------+----------+-----------------+---------+---------+-----------+---------------+----------------+--------+
|               284303|        69|               10|    976.0|  59568.0|           |               |      2015-12-02|   -10.0|
|               175408|       210|               10|    894.0|  36730.0|           |               |      2014-11-19|   -10.0|
|               175498|        26|               10|    173.0|  36750.0|           |               |      2014-11-19|   -10.0|
|               273388|        65|               10|    111.0|  57285.0|           |               |      2015-10-23|   -10.0|
|               235320|         9|               10|    864.0|  49272.0|           |               |      2015-

In [16]:
# Extender M_Fecha hasta 2016-05-31
from datetime import datetime, timedelta

fecha_inicio = datetime(2015, 10, 1)  # desde donde quedó
fecha_fin = datetime(2016, 5, 31)

fechas_adicionales = []
# El ultimo ID_Fecha fue 639, continuamos desde 640
id_fecha = 640

while fecha_inicio <= fecha_fin:
    fechas_adicionales.append((
        id_fecha,
        fecha_inicio.strftime('%Y-%m-%d'),
        fecha_inicio.day,
        fecha_inicio.strftime('%B'),
        fecha_inicio.year,
        int(fecha_inicio.strftime('%V'))
    ))
    fecha_inicio += timedelta(days=1)
    id_fecha += 1

schema_fecha = ['ID_Fecha', 'Fecha', 'Dia', 'Mes', 'Anio', 'Numero_semana_ISO']
df_fechas_adicionales = spark.createDataFrame(fechas_adicionales, schema=schema_fecha)

print("Fechas adicionales generadas:", df_fechas_adicionales.count())
df_fechas_adicionales.show(5)

# CARGUE
guardar_df(df_fechas_adicionales, 'DB_202613_n_cantini.M_Fecha')
print("Fechas adicionales cargadas correctamente")

Fechas adicionales generadas: 244
+--------+----------+---+-------+----+-----------------+
|ID_Fecha|     Fecha|Dia|    Mes|Anio|Numero_semana_ISO|
+--------+----------+---+-------+----+-----------------+
|     640|2015-10-01|  1|October|2015|               40|
|     641|2015-10-02|  2|October|2015|               40|
|     642|2015-10-03|  3|October|2015|               40|
|     643|2015-10-04|  4|October|2015|               40|
|     644|2015-10-05|  5|October|2015|               41|
+--------+----------+---+-------+----+-----------------+
only showing top 5 rows



Fechas adicionales cargadas correctamente


In [17]:
# JOINS CON DIMENSIONES PARA OBTENER IDs DWH
# Cargar dimensiones ya creadas desde la bodega
def obtener_df_destino(sql):
    return spark.read.format('jdbc') \
        .option('url', dest_db) \
        .option('dbtable', sql) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .load()

dim_proveedor = obtener_df_destino('DB_202613_n_cantini.M_Proveedor')
dim_tipo = obtener_df_destino('DB_202613_n_cantini.M_TipoTransaccion')
dim_fecha = obtener_df_destino('DB_202613_n_cantini.M_Fecha')

# Convertir tipos para hacer joins correctamente
movimientos = movimientos.withColumn('ProveedorID', col('ProveedorID').cast('int'))
movimientos = movimientos.withColumn('ClienteID', col('ClienteID').cast('int'))
movimientos = movimientos.withColumn('InvoiceID', col('InvoiceID').cast('int'))
movimientos = movimientos.withColumn('OrdenDeCompraID', col('OrdenDeCompraID').cast('int'))
movimientos = movimientos.withColumn('Cantidad', col('Cantidad').cast('int'))

# Hacer joins
hechos = movimientos.alias('m') \
    .join(dim_proveedor.alias('p'), 
          col('m.ProveedorID') == col('p.ID_Proveedor_T'), 'left') \
    .join(dim_tipo.alias('t'), 
          col('m.TipoTransaccionID') == col('t.ID_TipoTransaccion_T'), 'left') \
    .join(dim_fecha.alias('f'), 
          col('m.FechaTransaccion') == col('f.Fecha'), 'left') \
    .select(
        col('m.TransaccionProductoID').alias('ID_Movimiento'),
        col('m.ProductoID').alias('ID_Producto_DWH'),
        col('m.ClienteID').alias('ID_Cliente_DWH'),
        col('p.ID_Proveedor_DWH'),
        col('t.ID_TipoTransaccion_DWH'),
        col('f.ID_Fecha'),
        col('m.InvoiceID').alias('Invoice_ID'),
        col('m.OrdenDeCompraID').alias('OrdenCompra_ID'),
        col('m.Cantidad')
    ) \
    .fillna({
        'ID_Proveedor_DWH': 0,
        'ID_TipoTransaccion_DWH': 0,
        'ID_Fecha': 0
    })

print("Total registros tabla de hechos:", hechos.count())
hechos.show(5)

Total registros tabla de hechos: 173659


+-------------+---------------+--------------+----------------+----------------------+--------+----------+--------------+--------+
|ID_Movimiento|ID_Producto_DWH|ID_Cliente_DWH|ID_Proveedor_DWH|ID_TipoTransaccion_DWH|ID_Fecha|Invoice_ID|OrdenCompra_ID|Cantidad|
+-------------+---------------+--------------+----------------+----------------------+--------+----------+--------------+--------+
|       284303|             69|           976|               0|                     9|     702|     59568|          NULL|     -10|
|       175408|            210|           894|               0|                     9|     324|     36730|          NULL|     -10|
|       175498|             26|           173|               0|                     9|     324|     36750|          NULL|     -10|
|       273388|             65|           111|               0|                     9|     662|     57285|          NULL|     -10|
|       235320|              9|           864|               0|                    

In [18]:
# CARGUE
guardar_df(hechos, 'DB_202613_n_cantini.M_Movimiento')
print("M_Movimiento cargada correctamente")

M_Movimiento cargada correctamente
